# Ngày 1 — Kiểm tra dữ liệu giá

Mục tiêu: xác nhận dữ liệu đã nạp vào kho là **đúng** trước khi làm bất kỳ phân tích nào.

Chạy trước trong terminal:
```
python scripts/make_sample_data.py
python -m vnsa.cli load --sample
```
Khi đã có file CSV thật trong `data/raw/prices/`, chạy `python -m vnsa.cli load` và đổi `SYMBOL` bên dưới.

In [ ]:
from vnsa.config import path
from vnsa.data.store import Store
from vnsa.data.cleaning import validate_prices, summarize
from vnsa.report.charts import candlestick

SYMBOL = "DEMO_A"   # đổi thành FPT, HPG... khi có dữ liệu thật

with Store(path("warehouse")) as store:
    display(store.inventory())
    df = store.prices(SYMBOL)
df.tail()

## 1. Kiểm tra chất lượng

In [ ]:
print(summarize(df))
issues = validate_prices(df)
print("\n".join(map(str, issues)) or "Không phát hiện vấn đề.")

## 2. Kiểm tra bằng mắt

Nhìn kỹ: có khoảng trống bất thường không, có phiên nào giá "rơi thẳng đứng" (thường là ngày GDKHQ chưa điều chỉnh) không, MA200 có hợp lý không.

In [ ]:
candlestick(df[df["date"] >= "2024-01-01"]).show()

## 3. Thống kê nhanh lợi suất

Đây là nền cho phần Quant sau này: lợi suất log, biến động năm hoá, drawdown lớn nhất.

In [ ]:
import numpy as np

r = np.log(df["close"]).diff().dropna()
equity = df["close"] / df["close"].cummax()
print(f"Lợi suất TB năm hoá : {r.mean() * 250:.1%}")
print(f"Biến động năm hoá   : {r.std() * np.sqrt(250):.1%}")
print(f"Ngày giảm mạnh nhất : {r.min():.2%}  |  tăng mạnh nhất: {r.max():.2%}")
print(f"Max drawdown        : {equity.min() - 1:.1%}")